In [ ]:
import os
# Preload CUDA runtime DLLs to avoid WinError 1114
_cuda_bin = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0\bin'
if os.path.isdir(_cuda_bin):
    os.add_dll_directory(_cuda_bin)
    if os.path.isdir(_cuda_bin + r'\x64'):
        os.add_dll_directory(_cuda_bin + r'\x64')

import numpy as np
import math
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, SequentialLR, LinearLR
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def set_seed(seed=2025):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

In [ ]:
MANIFEST_PATH = 'ScAlN_dataset_3class/split_manifest.json'

def load_data(data_path='ScAlN_dataset_3class',
              manifest_path=MANIFEST_PATH):
    print(f"Loading splits from: {manifest_path}")
    
    X_data = np.load(os.path.join(data_path, 'X_ScAlN_3class.npy'))
    y_data = np.load(os.path.join(data_path, 'y_ScAlN_3class_labeled.npy'))
    
    print(f"Raw data: X={X_data.shape}, y={y_data.shape}")
    print(f"Data range: [{X_data.min():.2f}, {X_data.max():.2f}]")
    
    if y_data.ndim > 1:
        y_data = y_data.flatten()
    
    if X_data.ndim == 3:
        X_data = np.expand_dims(X_data, axis=1)
    if X_data.shape[1] == 1:
        X_data = np.repeat(X_data, 3, axis=1)
    
    X_data = X_data.astype(np.float32)
    if X_data.max() > 1.0:
        X_data = X_data / 255.0
    
    print(f"Processed: X={X_data.shape}, range=[{X_data.min():.2f}, {X_data.max():.2f}]")
    
    # Label remap (1,2,3 -> 0,1,2)
    y_data = y_data - 1 if y_data.min() == 1 else y_data
    
    unique, counts = np.unique(y_data, return_counts=True)
    print(f"Class distribution: {dict(zip(unique, counts))}")
    
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
    
    label_names = ['Streaky', 'Transition', 'Spotty']
    splits = {}
    for split_name in ['train', 'val', 'test']:
        idx = manifest['splits'][split_name]['frame_indices']
        X = torch.FloatTensor(X_data[idx])
        y = torch.LongTensor(y_data[idx])
        splits[split_name] = (X, y)
        u, c = np.unique(y_data[idx], return_counts=True)
        dist = {label_names[k]: f"{v}({v/len(idx)*100:.1f}%)" for k, v in zip(u, c)}
        print(f"{split_name} distribution: {dist}, total {len(idx)} frames")
    
    # Class weights based on training set
    train_y = y_data[manifest['splits']['train']['frame_indices']]
    class_weights = compute_class_weight('balanced', classes=np.unique(train_y), y=train_y)
    print(f"Class weights: {dict(zip(unique, class_weights))}")
    
    return splits['train'], splits['val'], splits['test'], torch.FloatTensor(class_weights)

In [ ]:
class RHEEDDataset(Dataset):
    def __init__(self, X, y, training=True):
        self.X, self.y = X, y
        self.training = training
        
    def __getitem__(self, idx):
        img = self.X[idx].clone()
        if self.training:
            # Random brightness (simulate exposure variation)
            if torch.rand(1) < 0.3:
                img = img * (0.8 + 0.4 * torch.rand(1).item())
            # Gaussian noise (simulate sensor noise)
            if torch.rand(1) < 0.3:
                img = img + torch.randn_like(img) * 0.02
            # Random erasing (simulate local occlusion/noise)
            if torch.rand(1) < 0.2:
                _, h, w = img.shape
                eh = int(h * (0.05 + 0.1 * torch.rand(1).item()))
                ew = int(w * (0.05 + 0.1 * torch.rand(1).item()))
                y0 = torch.randint(0, h - eh, (1,)).item()
                x0 = torch.randint(0, w - ew, (1,)).item()
                img[:, y0:y0+eh, x0:x0+ew] = torch.rand(3, eh, ew)
            img = torch.clamp(img, 0, 1)
        return img, self.y[idx]
    
    def __len__(self):
        return len(self.X)

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, in_channels=3, dr=0.3):
        super().__init__()
        self.pad1 = nn.ZeroPad2d((2, 2, 0, 0))  # pad 2 on left/right
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1, 3))
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d((1, 2))
        
        self.pad2 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(2, 3))
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d((1, 2))
        
        self.pad3 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv3 = nn.Conv2d(128, 256, kernel_size=(2, 3))
        self.bn3 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d((1, 2))
        
        self.pad4 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv4 = nn.Conv2d(256, 256, kernel_size=(1, 3))
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d((1, 2))
        
        self.dropout = nn.Dropout(dr)
        
    def forward(self, x):
        x = self.pad1(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(self.pool1(x))
        
        x = self.pad2(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(self.pool2(x))
        
        x = self.pad3(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(self.pool3(x))
        
        x = self.pad4(x)
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.dropout(self.pool4(x))
        
        return x

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, num_heads=4, mlp_ratio=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout)
        )
        
    def forward(self, x):
        x1 = self.norm1(x)
        attn_out, _ = self.attn(x1, x1, x1)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class PatchEncoder(nn.Module):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = nn.Linear(256, projection_dim)  # CNN outputs 256 channels
        self.position_embedding = nn.Embedding(num_patches, projection_dim)
        
    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device)
        return self.projection(x) + self.position_embedding(positions)

In [ ]:
class CNNTransformer(nn.Module):
    def __init__(self, num_classes=3, dr=0.3, projection_dim=64, 
                 num_heads=4, transformer_layers=8,
                 input_shape=(3, 32, 64)):
        super().__init__()
        self.cnn = CNNFeatureExtractor(in_channels=input_shape[0], dr=dr)
        self.projection_dim = projection_dim
        
        # infer flatten dim
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            cnn_out = self.cnn(dummy)
            _, C, H, W = cnn_out.shape
            num_patches = H * W
            flatten_dim = num_patches * projection_dim
        
        self.patch_proj = nn.Linear(C, projection_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, projection_dim) * 0.02)
        
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(projection_dim, num_heads, mlp_ratio=2, dropout=0.1)
            for _ in range(transformer_layers)
        ])
        
        self.norm = nn.LayerNorm(projection_dim, eps=1e-6)
        self.dropout = nn.Dropout(0.3)
        
        # MLP head
        self.classifier = nn.Sequential(
            nn.Linear(flatten_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_classes)
        )
        
    def forward(self, x):
        features = self.cnn(x)  # [B, 256, H', W']
        B, C, H, W = features.shape
        
        # Reshape to sequence [B, H*W, C]
        features = features.view(B, C, H * W).transpose(1, 2)
        
        features = self.patch_proj(features) + self.pos_embed
        
        for block in self.transformer_blocks:
            features = block(features)
        
        features = self.norm(features)
        features = self.dropout(features)
        features = features.flatten(1)
        return self.classifier(features)

In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes=3, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.weight = weight
        
    def forward(self, pred, target):
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        
        log_prob = F.log_softmax(pred, dim=1)
        loss = -true_dist * log_prob
        
        if self.weight is not None:
            w = self.weight.to(pred.device)[target]
            loss = loss.sum(1) * w
        else:
            loss = loss.sum(1)
        return loss.mean()

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += targets.size(0)
        correct += pred.eq(targets).sum().item()
    
    return total_loss/len(loader), correct/total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    class_correct = [0, 0, 0]
    class_total = [0, 0, 0]
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()
            
            for i in range(3):
                class_total[i] += (targets == i).sum().item()
                class_correct[i] += ((pred == i) & (targets == i)).sum().item()
    
    recalls = [class_correct[i]/max(class_total[i], 1) for i in range(3)]
    return total_loss/len(loader), correct/total, recalls

In [ ]:
def main():
    set_seed(2025)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    
    # Video-segment-level split via split_manifest.json
    (X_train, y_train), (X_val, y_val), (X_test, y_test), class_weights = load_data()
    
    train_dataset = RHEEDDataset(X_train, y_train, training=True)
    val_dataset = RHEEDDataset(X_val, y_val, training=False)
    test_dataset = RHEEDDataset(X_test, y_test, training=False)
    
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, pin_memory=True)
    
    print(f"Datasets: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
    print(f"Augmentation: enabled (brightness, Gaussian noise, random erasing)")
    
    model = CNNTransformer(
        num_classes=3,
        dr=0.3,
        projection_dim=64,
        num_heads=4,
        transformer_layers=8,
        input_shape=(3, 32, 64)
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal params:   {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable params: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    
    criterion = LabelSmoothingCE(num_classes=3, smoothing=0.1, weight=class_weights)
    
    optimizer = optim.Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-7)
    
    # LR schedule: 5-epoch warm-up + cosine annealing
    epochs = 130
    warmup_epochs = 5
    steps_per_epoch = len(train_loader)
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs * steps_per_epoch)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=(epochs - warmup_epochs) * steps_per_epoch, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], 
                             milestones=[warmup_epochs * steps_per_epoch])
    
    best_val_acc = 0
    patience, counter = 35, 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print(f"\nStarting training...")
    print(f"Config: epochs={epochs}, label_smoothing=0.1, warmup={warmup_epochs}ep, cosine_annealing, augmentation enabled")
    train_start_time = time.time()
    
    for epoch in range(epochs):
        # Train, update LR every step
        model.train()
        total_loss, correct, total = 0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()
        
        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        
        val_loss, val_acc, recalls = validate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model_improved.pth')
            counter = 0
        else:
            counter += 1
        
        cur_lr = optimizer.param_groups[0]['lr']
        if (epoch + 1) % 10 == 0 or epoch < warmup_epochs:
            print(f'Epoch [{epoch+1}/{epochs}] Train: {train_acc:.4f} | Val: {val_acc:.4f} | '
                  f'Recalls: {recalls[0]:.3f}/{recalls[1]:.3f}/{recalls[2]:.3f} | LR: {cur_lr:.2e}')
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    training_time = time.time() - train_start_time
    print(f"\nTotal training time: {training_time:.1f}s ({training_time/60:.1f}min)")
    print(f"Epochs trained: {len(history['train_loss'])} epochs")
    
    model.load_state_dict(torch.load('best_model_improved.pth'))
    test_loss, test_acc, test_recalls = validate(model, test_loader, criterion, device)
    
    print(f"\n{'='*60}")
    print(f"Training complete - final results")
    print(f"{'='*60}")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"Per-class recall: {test_recalls}")
    
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs.to(device))
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
    
    print("\nClassification report:")
    print(classification_report(all_targets, all_preds, 
                                target_names=['Streaky', 'Transition', 'Spotty']))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    cm = confusion_matrix(all_targets, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=['Streaky', 'Transition', 'Spotty'],
                yticklabels=['Streaky', 'Transition', 'Spotty'])
    axes[2].set_title('Confusion Matrix')
    
    plt.tight_layout()
    plt.savefig('CNN_Transformer_PyTorch_improved.png', dpi=300)
    plt.show()
    
    return history, all_targets, all_preds, total_params, trainable_params, training_time

In [ ]:
if __name__ == "__main__":
    history, all_targets, all_preds, total_params, trainable_params, training_time = main()